In [ ]:
!pip install rasterio geopandas

In [ ]:
import os
import glob
import rasterio
from rasterio.mask import mask
import geopandas as gpd

In [ ]:
# --- Files path ---
raster_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory"
shapefile_path = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\hungary_shape\hun_countrybuff.shp"
output_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Clipped"

os.makedirs(output_folder, exist_ok=True)


In [ ]:
print("Nodata:", src.nodata)
print("Dtype:", src.dtypes)

In [ ]:
print("Raster CRS:", src.crs)
print("Shape CRS:", gdf.crs)

In [ ]:
# --- Loading shapefile ---
gdf = gpd.read_file(shapefile_path)

In [ ]:
#-- Checking for .tif files in subfolders--
from pathlib import Path

raster_files = list(Path(raster_folder).rglob("*.tif"))
print(len(raster_files))


In [ ]:
# --- Raster files recursive listing ---
raster_files = glob.glob(os.path.join(raster_folder, "**", "*.tif"), recursive=True)

print(f"Raster files found: {len(raster_files)}")

for raster_path in raster_files:
    
    with rasterio.open(raster_path) as src:
        
        # CRS check
        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)
        
        geometries = gdf.geometry.values
        
        # Clip
        out_image, out_transform = mask(src, geometries, crop=True, filled=True, nodata=src.nodata)
        
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })
        
        # Only filename left behind
        out_path = os.path.join(output_folder, os.path.basename(raster_path))
        
        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(out_image)

print("Done.")

In [ ]:
#-- Csak nyári hónapok leválogatása és másik mappába másolása
import os
from pathlib import Path

# Mappa
raster_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Clipped"

# csak .tif-ek
all_files = list(Path(raster_folder).rglob("*.tif"))

summer_months = ["05", "06", "07", "08", "09"]
summer_files = []

for f in all_files:
    fname = f.name
    # feltételezzük, hogy YYYYMMDD a fájlnévben
    date_str = fname.split("_")[3]  # pl. "20200101"
    month = date_str[4:6]           # 05
    if month in summer_months:
        summer_files.append(f)

print(f"Összes fájl: {len(all_files)}")
print(f"Nyári hónapos fájlok: {len(summer_files)}")

# Ha akarsz, kimentheted másik mappába
output_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Summer_Rasters"
os.makedirs(output_folder, exist_ok=True)

for f in summer_files:
    dest = Path(output_folder) / f.name
    # másolás
    import shutil
    shutil.copy(f, dest)